# Dummy XGBoost smoke test

End-to-end check of `ModelTrainingWorkflow`: 1000-row parquet, temporal train/validation/test split, a small XGBoost model.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). That server proxies artifacts into the MinIO bucket `s3://mlflow/` (`http://localhost:9000`).

In [ ]:
from pathlib import Path
import os

import mlflow
import numpy as np
import pandas as pd

from src.run_time_configuration import BaseConfigParams
from src.model_training.training import ModelTrainingWorkflow

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DUMMY_PATH = DATA_DIR / "dummy_fraud_1000.parquet"

TARGET_COLUMN = "Flag_NotPaid_Within_90_Days_After_DueDate"
N_ROWS = 1000
N_TRAIN_PERIOD = 800
N_TEST_PERIOD = 200
SEED = 42


def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")



MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dummy parquet: {DUMMY_PATH}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

ModuleNotFoundError: No module named 'modeling_fraud_system.src'

In [ ]:
rng = np.random.default_rng(SEED)

train_start = pd.Timestamp("2024-01-01", tz="UTC")
train_end = pd.Timestamp("2025-06-30 23:59:59", tz="UTC")
test_start = pd.Timestamp("2025-07-01", tz="UTC")
test_end = pd.Timestamp("2025-08-31 23:59:59", tz="UTC")

train_offsets = rng.integers(0, int((train_end - train_start).total_seconds()) + 1, size=N_TRAIN_PERIOD)
test_offsets = rng.integers(0, int((test_end - test_start).total_seconds()) + 1, size=N_TEST_PERIOD)
request_datetime = pd.concat(
    [
        pd.Series(train_start + pd.to_timedelta(train_offsets, unit="s")),
        pd.Series(test_start + pd.to_timedelta(test_offsets, unit="s")),
    ],
    ignore_index=True,
)

transaction_amt = rng.lognormal(mean=3.5, sigma=0.8, size=N_ROWS)
account_age_days = rng.integers(1, 2000, size=N_ROWS).astype(float)
n_prev_transactions = rng.poisson(8, size=N_ROWS).astype(float)
distance = rng.exponential(50, size=N_ROWS)
hour_of_day = rng.integers(0, 24, size=N_ROWS).astype(float)
n_failed_attempts = rng.poisson(0.4, size=N_ROWS).astype(float)
email_risk_score = rng.beta(2, 8, size=N_ROWS)
device_mismatch = rng.binomial(1, 0.15, size=N_ROWS).astype(float)
product_cd = rng.choice(["W", "C", "H", "R", "S"], size=N_ROWS)

logit = (
    -2.2
    + 0.02 * (transaction_amt - 30)
    - 0.001 * account_age_days
    + 0.45 * n_failed_attempts
    + 2.2 * email_risk_score
    + 0.9 * device_mismatch
    + 0.004 * distance
    + 0.35 * (hour_of_day < 6)
)
prob = 1.0 / (1.0 + np.exp(-logit))
y = rng.binomial(1, np.clip(prob, 0.03, 0.55), size=N_ROWS)

dummy_df = pd.DataFrame(
    {
        "RequestDateTime": request_datetime,
        "transaction_amt": transaction_amt,
        "account_age_days": account_age_days,
        "n_prev_transactions": n_prev_transactions,
        "distance": distance,
        "hour_of_day": hour_of_day,
        "n_failed_attempts": n_failed_attempts,
        "email_risk_score": email_risk_score,
        "device_mismatch": device_mismatch,
        "ProductCD": product_cd,
        TARGET_COLUMN: y,
    }
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

dummy_df.to_parquet(DUMMY_PATH, engine="pyarrow", index=False)

print(dummy_df.shape)
print(dummy_df.dtypes)
print(dummy_df[TARGET_COLUMN].value_counts(normalize=True).round(3))
print(
    dummy_df.assign(period=np.where(dummy_df["RequestDateTime"] < test_start, "train_val", "test"))
    .groupby("period")
    .size()
)
dummy_df.head()

In [ ]:
runtime_params = BaseConfigParams(
    experiment_name="dummy_fraud_xgboost",
    mlflow_run_name="dummy_xgb_simple",
    training_start_date="2024-01-01",
    training_end_date="2025-06-30",
    test_start_date="2025-07-01",
    test_end_date="2025-08-31",
    validation_start_date="2024-01-01",
    validation_end_date="2025-06-30",
)

workflow = ModelTrainingWorkflow(runtime_params)
workflow.load_data(path_external_data=str(DUMMY_PATH), columns=list(dummy_df.columns))
workflow.split_data(target_column=TARGET_COLUMN, validation_split=0.2)
workflow.generate_preprocessed_data(numeric_only=True)

print("train", workflow.x_train_preproc.shape, "pos_rate", float(workflow.y_train.mean()))
print("valid", workflow.x_validation_preproc.shape, "pos_rate", float(workflow.y_validation.mean()))
print("test ", workflow.x_test_preproc.shape, "pos_rate", float(workflow.y_test.mean()))
print("numeric features", list(workflow.x_train_preproc.columns))

In [ ]:
results = workflow.train_model(
    hyperparameters={
        "n_estimators": 40,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="dummy_xgb_simple",
)

metrics_table = pd.DataFrame(
    {
        "train": results["train_metrics"],
        "validation": results["val_metrics"],
        "test": results["test_metrics"],
    }
).loc[["accuracy", "precision", "recall", "f1", "auc", "pr_auc", "ks_statistic"]]

run_id = workflow.run_time_config.mlflow_run_id
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

print("threshold", round(results["threshold"], 4))
print("mlflow run id", run_id)
print("tracking uri", mlflow.get_tracking_uri())
print("artifact uri", run.info.artifact_uri)
print("logged artifacts", [a.path for a in client.list_artifacts(run_id)])
metrics_table.round(4)